# Diagnóstico de calidad de datos: criminalidad en Medellín

**Evidencia:** GA1-220501094-01-AA1-EV06
**Aprendiz:** Liliana Sofia Panesso Mendez

## Momento 1. Reconocimiento del proyecto y de los datasets

**Proyecto:** Análisis de los delitos más frecuentes en Medellín y su variación a través del tiempo.

**Propósito:** Conocer el comportamiento de la criminalidad en Medellín a partir de registros oficiales verificables, y no solo de percepciones, identificando qué delitos tienen más casos registrados y cómo ha sido su tendencia. Antes de iniciar ese análisis, este notebook diagnostica la calidad de los datasets seleccionados para determinar si están en condiciones adecuadas de uso y qué acciones de preparación requieren.

**Pregunta analítica:** ¿Cuáles son los delitos con mayor cantidad de casos registrados en Medellín y cómo ha variado su número de casos entre 2020 y 2025?

### Datasets seleccionados

| # | Dataset | Fuente | Qué contiene | Relación con la pregunta |
|---|---|---|---|---|
| 1 | criminalidad_anio_mes.csv | MEData, Alcaldía de Medellín (Secretaría de Seguridad y Convivencia). Republicado en Datos Abiertos Colombia. Licencia CC BY-SA 4.0 | Cantidad de casos de 7 conductas delictivas (extorsión, homicidio y hurtos a persona, de carro, de moto, a residencia y de semoviente) por año y mes, desde 2003. Columnas: `Fecha_hecho`, `Conducta`, `Cantidad_casos` | **Dataset principal.** Permite identificar los delitos con más casos y analizar su variación entre 2020 y 2025 |
| 2 | criminalidad_comunas_anio.csv | MEData, Alcaldía de Medellín (Secretaría de Seguridad y Convivencia). Republicado en Datos Abiertos Colombia. Licencia CC BY-SA 4.0 | Cantidad de casos de las mismas 7 conductas por comuna y año, desde 2003. Columnas: `Fecha_hecho`, `Conducta`, `Codigo_comuna`, `Cantidad_casos` | **Dataset complementario.** Aporta el lugar de ocurrencia para comparar los delitos entre zonas de la ciudad y contrastar los totales con el dataset 1 |
| 3 | homicidios_banco_mundial.csv | Banco Mundial, Indicadores del desarrollo mundial (datos de UNODC). Archivo original: `API_VC.IHR.PSRC.P5_DS2_es_csv_v2_13913.csv` | Tasa de homicidios intencionales por cada 100.000 habitantes, por país y región, con una columna por año desde 1960 | **Dataset de contexto.** No responde la pregunta por sí solo, pero permite ubicar la tendencia de homicidios de Colombia en el mismo período |

### Aspectos de calidad a revisar

1. **Cobertura temporal:** verificar que los años 2020 a 2025 estén completos en cada dataset, especialmente 2025.
2. **Valores faltantes:** identificar celdas vacías y valores faltantes "disfrazados" de texto, como `SIN DATO` en la columna de comunas.
3. **Registros duplicados:** comprobar que no haya filas repetidas que inflen los conteos.
4. **Tipos de datos:** revisar que las fechas, los años y las cantidades tengan un tipo coherente con lo que representan.
5. **Consistencia de categorías:** confirmar que los nombres de las conductas estén escritos de la misma forma dentro de cada dataset y entre los datasets 1 y 2.
6. **Valores incoherentes:** buscar cantidades negativas, años fuera de rango o códigos de comuna inválidos.
7. **Coherencia entre datasets:** comparar el formato de `Fecha_hecho` (año y mes frente a solo año) y verificar si los totales por comuna coinciden con los totales del dataset 1.
8. **Formato del archivo:** el archivo del Banco Mundial trae líneas de metadatos antes del encabezado, una columna por año y una columna final vacía.
9. **Comparabilidad de la medida:** el Banco Mundial reporta una tasa nacional por 100.000 habitantes, mientras que los datasets de Medellín reportan cantidad de casos.
10. **Limitación de origen:** los registros corresponden a delitos denunciados, por lo que existe subregistro de los casos no reportados.

## Momento 2. Carga de los datasets

In [1]:
import pandas as pd

crim_mes = pd.read_csv("../data/raw/criminalidad_anio_mes.csv", encoding="utf-8")
crim_comunas = pd.read_csv("../data/raw/criminalidad_comunas_anio.csv", encoding="utf-8")
homicidios_bm = pd.read_csv("../data/raw/homicidios_banco_mundial.csv", encoding="utf-8", skiprows=4)

print("Datasets cargados correctamente")

Datasets cargados correctamente


In [2]:
print("crim_mes:", crim_mes.shape)
print("crim_comunas:", crim_comunas.shape)
print("homicidios_bm:", homicidios_bm.shape)

crim_mes: (1586, 3)
crim_comunas: (2569, 4)
homicidios_bm: (266, 71)


**Verificación de la carga:** los tres datasets se cargaron sin errores. El número de filas coincide con las líneas de cada archivo (descontando el encabezado y, en el caso del Banco Mundial, las 4 líneas de metadatos omitidas con `skiprows=4`). El dataset del Banco Mundial presenta 71 columnas: 4 descriptivas, 66 de años (1960–2025) y una columna adicional generada por una coma sobrante al final de cada línea.